# 01 — Data Exploration

**NewsBot Intelligence System 2.0** | ITAI 2373 | Trilok Kalani (SOLO)

Load the BBC News corpus and look at its shape, class balance, and article length before modeling.

In [1]:
# --- Setup: works in Colab and locally ---
import os, sys, subprocess

def find_repo_root(start="."):
    p = os.path.abspath(start)
    for _ in range(6):
        if os.path.isdir(os.path.join(p, "src")) and os.path.exists(os.path.join(p, "src", "newsbot.py")):
            return p
        p = os.path.dirname(p)
    return None

ROOT = find_repo_root()
if ROOT is None:
    # Running on a fresh Colab: clone the repo
    if not os.path.isdir("ITAI2373-Portfolio"):
        subprocess.run(["git","clone","--depth","1",
                        "https://github.com/Tikskalani/ITAI2373-Portfolio.git"], check=False)
    ROOT = find_repo_root("ITAI2373-Portfolio/ITAI2373-NewsBot-Final") or \
           find_repo_root("ITAI2373-Portfolio")
sys.path.insert(0, ROOT)
print("Repo root:", ROOT)

# spaCy model (quiet no-op if already present)
try:
    import spacy; spacy.load("en_core_web_sm")
except Exception:
    subprocess.run([sys.executable,"-m","spacy","download","en_core_web_sm"], check=False)

import pandas as pd
DATA = os.path.join(ROOT, "data", "raw", "newsbot_bbc.csv")
df = pd.read_csv(DATA)
print("Loaded", len(df), "articles;", df["category"].nunique(), "categories")
df.head(2)

Repo root: /content/ITAI2373-NewsBot-Final


Loaded 2225 articles; 5 categories


,article_id,category,text
0,business_001,business,Ad sales boost Time Warner profit Quarterly pr...
1,business_002,business,Dollar gains on Greenspan speech The dollar ha...


### Class balance
The five categories are reasonably balanced, which matters for a fair classifier.

In [2]:
counts = df["category"].value_counts()
print(counts)
counts.plot(kind="bar", title="Articles per category", color="#1f2a44"); 

category
sport            511
business         510
politics         417
tech             401
entertainment    386
Name: count, dtype: int64


### Article length
Most articles sit in a few hundred words. Length varies by beat.

In [3]:
df["n_words"] = df["text"].str.split().str.len()
print(df.groupby("category")["n_words"].median())
df["n_words"].plot(kind="hist", bins=40, title="Article length (words)", color="#f97316");

category
business         297.0
entertainment    262.5
politics         439.0
sport            288.0
tech             447.0
Name: n_words, dtype: float64


### A sample article

In [4]:
row = df.sample(1, random_state=1).iloc[0]
print("CATEGORY:", row["category"], "\n")
print(row["text"][:600], "...")

CATEGORY: sport 

Wood - Ireland can win Grand Slam Former captain Keith Wood believes Ireland can win only their second Grand Slam - and first since 1948 - in this year's RBS Six Nations Championship. After claiming their first Triple Crown for 19 years last season, Wood tips his former team-mates to go one better. "Things have been building up over the past few years and I think this is the year for Ireland," he told BBC Sport. "There is a great chance to win a Grand Slam. A lot of things are in our favour with England and France at home." Ireland have finished runners-up three times, including last year, sin ...


**Takeaway.** The corpus is clean, balanced, and full-text, so it supports both supervised classification and unsupervised topic modeling downstream.